# 01. Dataset quality and prediction-time contract

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd()
if not (ROOT / "src").exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.config import RESULTS_DIR, ALL_MODEL_FEATURES, FORBIDDEN_COLUMNS
from src.data import load_raw_dataset, clean_and_prepare_dataset, attach_label_availability
from src.features import compute_schedule_and_calendar_features
summary = json.loads((RESULTS_DIR / "run_summary.json").read_text())
plt.rcParams.update({"figure.figsize": (9, 4), "font.size": 11})


We predict arrival delay of at least 15 minutes, two hours before scheduled departure. The supervised population requires an observed arrival outcome. Missing outcomes are not labeled on time.

In [2]:
raw = load_raw_dataset()
clean, profile = clean_and_prepare_dataset(raw)
print("Raw shape:", raw.shape, "Observed arrival outcomes:", len(clean))
print("Duplicate flight keys:", profile["duplicate_flight_keys"])
display(raw.head(3))

Raw shape: (336776, 21) Observed arrival outcomes: 327346
Duplicate flight keys: 0


,id,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,...,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour,name
0,0,2013,1,1,517.0,515,2.0,830.0,819,11.0,...,1545,N14228,EWR,IAH,227.0,1400,5,15,2013-01-01 05:00:00,United Air Lines Inc.
1,1,2013,1,1,533.0,529,4.0,850.0,830,20.0,...,1714,N24211,LGA,IAH,227.0,1416,5,29,2013-01-01 05:00:00,United Air Lines Inc.
2,2,2013,1,1,542.0,540,2.0,923.0,850,33.0,...,1141,N619AA,JFK,MIA,160.0,1089,5,40,2013-01-01 05:00:00,American Airlines Inc.


In [3]:
missing = raw.isna().sum().rename("missing_rows").to_frame()
missing["missing_fraction"] = missing["missing_rows"] / len(raw)
display(missing[missing.missing_rows > 0])
display(pd.Series(profile).drop(["mapped_columns", "raw_missing_counts"]).to_frame("value"))

,missing_rows,missing_fraction
dep_time,8255,0.024512
dep_delay,8255,0.024512
arr_time,8713,0.025872
arr_delay,9430,0.028001
tailnum,2512,0.007459
air_time,9430,0.028001


,value
initial_raw_rows,336776
initial_raw_cols,21
duplicate_flight_keys,0
invalid_dates_excluded,0
invalid_distance_excluded,0
excluded_cancellations_or_missing_target,9430
operated_flights_count,327346
target_column,is_delayed
raw_target_column,arr_delay
delay_threshold_minutes,15.0


Schedule fields, calendar, route and distance are potential predictors. Actual departure/arrival fields control retrospective label construction and historical availability audits only. They never enter the model feature matrix.

In [4]:
from src.validation import audit_feature_matrix_for_forbidden_columns
print(audit_feature_matrix_for_forbidden_columns(ALL_MODEL_FEATURES))
display(pd.DataFrame({"allowed_model_features": pd.Series(ALL_MODEL_FEATURES), "forbidden_fields": pd.Series(FORBIDDEN_COLUMNS)}))

{'is_clean': True, 'total_features': 35, 'violations_found': [], 'checked_against_forbidden_count': 26}


,allowed_model_features,forbidden_fields
0,carrier,dep_time
1,origin,dep_delay
2,dest,arr_time
3,sched_dep_hour,arr_delay
4,sched_dep_min,air_time
5,sched_dep_min_of_day,flight
6,sin_sched_dep,tailnum
7,cos_sched_dep,id
8,sched_arr_hour,name
9,sched_arr_min,time_hour


Actual arrival equals inferred scheduled arrival UTC plus observed arrival delay. The audit verifies the resulting destination local clock and plausible ground time. The declared 24-hour reporting lag is an assumption because true publication timestamps are unavailable.

In [5]:
features = compute_schedule_and_calendar_features(clean)
features, availability = attach_label_availability(features)
display(pd.Series(availability).to_frame("value"))
known = features.label_available_at.notna()
assert (features.loc[known, "label_available_at"] > features.loc[known, "decision_time"]).all()
print("Every usable current-flight label is released after that flight's prediction time.")
display(features[["flight_date", "decision_time", "label_available_at"]].head())

,value
reporting_lag_hours,24
arrival_clock_mismatches,1
history_labels_excluded_by_timestamp_audit,1
history_labels_usable,327345
scheduled_arrival_date_assumption,First destination local schedule clock after d...
reporting_assumption,Arrival outcome becomes available 24h after re...


Every usable current-flight label is released after that flight's prediction time.


,flight_date,decision_time,label_available_at
0,2013-01-01,2013-01-01 08:15:00+00:00,2013-01-02 14:30:00+00:00
1,2013-01-01,2013-01-01 08:29:00+00:00,2013-01-02 14:50:00+00:00
2,2013-01-01,2013-01-01 08:40:00+00:00,2013-01-02 14:23:00+00:00
3,2013-01-01,2013-01-01 08:45:00+00:00,2013-01-02 14:04:00+00:00
4,2013-01-01,2013-01-01 09:00:00+00:00,2013-01-02 13:12:00+00:00


In [6]:
manifest = summary["split_manifest"]
display(pd.DataFrame({k: manifest[k] for k in ["train", "validation", "test"]}).T)
display(pd.Series(manifest["model_fit"]).to_frame("value"))
display(pd.DataFrame(manifest["development_blocks"]).T)

,num_dates,start_date,end_date,row_count,positive_count,positive_prevalence,date_proportion,row_proportion
train,255,2013-01-01,2013-09-12,228299,59160,0.259134,0.69863,0.697424
validation,55,2013-09-13,2013-11-06,50466,7893,0.156402,0.150685,0.154167
test,55,2013-11-07,2013-12-31,48581,13047,0.268562,0.150685,0.148409


,value
fit_available_at,2013-09-13T07:00:00+00:00
candidate_training_rows,228299
fitted_rows,227520
purged_unavailable_or_invalid_labels,779


,start_date,end_date,candidate_rows,usable_rows,purged_rows,labels_available_before,prevalence
calibration_fit,2013-09-13,2013-10-01,17398,16434,964,2013-10-02T07:00:00+00:00,0.11963
calibration_selection,2013-10-02,2013-10-19,16451,15765,686,2013-10-20T07:20:00+00:00,0.187504
threshold_validation,2013-10-20,2013-11-06,16617,15646,971,2013-11-07T08:00:00+00:00,0.173782


The dataset covers one year and departures from three NYC airports. Generalization to all US flights or current operations is unverified. Missing outcomes create a conditional target population. The original test period has already been inspected, so the final experiment is a corrected retrospective reevaluation.